# ChillChain grader: train a crop grading model

Trains **one crop** to tell **A / B / C / other** from phone photos, using transfer learning on MobileNetV3 (a small model that runs in the phone browser). At the end it downloads `grader_<crop>.zip` for the app.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then edit the settings cell below and choose **Runtime → Run all**. If Colab asks to restart after installing packages, click Restart and **Run all** again.

Photo rules and folder layout: see `ml/README.md`.

In [ ]:
# 1. Settings: change these two lines
CROP = "strawberry"                                        # crop key used in the app
ZIP_PATH = "/content/drive/MyDrive/ChillChain/strawberry.zip"  # your photos zip on Google Drive

CLASSES = ["A", "B", "C", "other"]   # folder names inside the zip
IMG = 224                            # image size the model sees
EPOCHS_HEAD, EPOCHS_FINE = 12, 12
SEED = 42

In [ ]:
# 2. Install the TensorFlow.js converter (do this first; restart if Colab asks)
!pip -q install tensorflowjs
import os, json, random, shutil, pathlib, zipfile
import numpy as np, tensorflow as tf
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

In [ ]:
# 3. Get the photos from Google Drive
from google.colab import drive
drive.mount("/content/drive")
RAW, SPLIT = pathlib.Path("/content/raw"), pathlib.Path("/content/split")
shutil.rmtree(RAW, ignore_errors=True); shutil.rmtree(SPLIT, ignore_errors=True)
with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(RAW)

EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
def images(d): return sorted(p for p in pathlib.Path(d).rglob("*") if p.suffix.lower() in EXT and not p.name.startswith("."))
def find_root(base, names):
    # the folder that directly contains all `names` (zips often add one extra top folder)
    for d in [base, *[p for p in base.rglob("*") if p.is_dir()]]:
        if all((d / n).is_dir() for n in names): return d
    return None

root = find_root(RAW, ["train", "test"])
if root:  # you kept a separate test set (best): use it as is, split train into train/val
    print("Using your own test set from:", root / "test")
    src_train, src_test = root / "train", root / "test"
else:
    src_train, src_test = find_root(RAW, CLASSES), None
    assert src_train, f"Could not find folders {CLASSES} in the zip"

counts = {}
for c in CLASSES:
    files = images(src_train / c); random.shuffle(files)
    test_files = images(src_test / c) if src_test else []
    if not src_test:  # 70 / 15 / 15 split
        n = len(files); n_test = n_val = max(1, int(n * 0.15))
        test_files, val_files, train_files = files[:n_test], files[n_test:n_test + n_val], files[n_test + n_val:]
    else:             # 85 / 15 split of train into train / val
        n_val = max(1, int(len(files) * 0.15))
        val_files, train_files = files[:n_val], files[n_val:]
    for part, fs in (("train", train_files), ("val", val_files), ("test", test_files)):
        (SPLIT / part / c).mkdir(parents=True, exist_ok=True)
        for i, f in enumerate(fs): shutil.copy(f, SPLIT / part / c / f"{i:05d}{f.suffix.lower()}")
    counts[c] = (len(train_files), len(val_files), len(test_files))
for c, (a, b, t) in counts.items(): print(f"{c:6s} train {a:4d} | val {b:4d} | test {t:4d}")
low = [c for c, (a, _, _) in counts.items() if a < 150]
if low: print("⚠ Few photos for", low, "- aim for 300+ per class for good accuracy.")

In [ ]:
# 4. Load the images
def load(part, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        SPLIT / part, labels="inferred", label_mode="int", class_names=CLASSES,
        image_size=(IMG, IMG), batch_size=32, shuffle=shuffle, seed=SEED)
train_ds, val_ds, test_ds = load("train", True), load("val", False), load("test", False)
AUTOTUNE = tf.data.AUTOTUNE
train_ds, val_ds, test_ds = (d.cache().prefetch(AUTOTUNE) for d in (train_ds, val_ds, test_ds))

# Balance the classes so a big class does not dominate
n = np.array([counts[c][0] for c in CLASSES], dtype=float)
class_weight = {i: float(n.sum() / (len(n) * max(1, v))) for i, v in enumerate(n)}
print("class weights:", class_weight)

In [ ]:
# 5. Build the model: random changes to photos (only while training) + pre-trained MobileNetV3 + a new A/B/C/other layer
augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
    tf.keras.layers.RandomBrightness(0.25),
    tf.keras.layers.RandomContrast(0.25),
], name="augment")
# MobileNetV3 includes its own pixel scaling, so the model takes plain 0–255 RGB pixels.
base = tf.keras.applications.MobileNetV3Large(
    input_shape=(IMG, IMG, 3), include_top=False, weights="imagenet", pooling="avg")
base.trainable = False

inputs = tf.keras.Input((IMG, IMG, 3), name="image")
x = augment(inputs)
x = base(x, training=False)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(len(CLASSES), activation="softmax", name="grade")(x)
model = tf.keras.Model(inputs, outputs)

stop = tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print("Stage 1: train only the new layer")
model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, class_weight=class_weight, callbacks=[stop])

In [ ]:
# 6. Fine-tune: let the top of MobileNetV3 adapt to your fruit (small learning rate)
base.trainable = True
for layer in base.layers[:-40]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print("Stage 2: fine-tune")
model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FINE, class_weight=class_weight, callbacks=[stop])

In [ ]:
# 7. The honest exam: photos the model has never seen
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

probs = model.predict(test_ds, verbose=0)
y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_pred, conf = probs.argmax(1), probs.max(1)
test_acc = float((y_pred == y_true).mean())
print(f"TEST ACCURACY: {test_acc:.1%} on {len(y_true)} photos\n")
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=3))

ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred, labels=range(len(CLASSES))), display_labels=CLASSES).plot(cmap="Greens")
plt.title(f"{CROP}: rows = true grade, columns = model's grade"); plt.show()

print("If the app only answers when the model is at least this sure:")
print("threshold | answers | accuracy on those answers")
for th in (0.5, 0.6, 0.7, 0.8, 0.9):
    keep = conf >= th
    acc = (y_pred[keep] == y_true[keep]).mean() if keep.any() else float("nan")
    print(f"   {th:.1f}    |  {keep.mean():5.0%}  |  {acc:.1%}")

In [ ]:
# 8. See the mistakes (add more photos like these, then run again)
wrong = np.where(y_pred != y_true)[0][:12]
# image_dataset_from_directory reads classes in CLASSES order and files sorted inside each class
ordered = [f for c in CLASSES for f in sorted(str(p) for p in (SPLIT / "test" / c).iterdir())]
plt.figure(figsize=(14, 8))
for i, k in enumerate(wrong):
    plt.subplot(3, 4, i + 1); plt.axis("off")
    plt.imshow(plt.imread(ordered[k]))
    plt.title(f"true {CLASSES[y_true[k]]} → said {CLASSES[y_pred[k]]} ({conf[k]:.0%})", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# 9. Export for the app (TensorFlow.js) and download
OUT = pathlib.Path("/content/export"); shutil.rmtree(OUT, ignore_errors=True); OUT.mkdir()
model.export(str(OUT / "saved_model"))  # inference-only graph: the random photo changes are switched off
!tensorflowjs_converter --input_format=tf_saved_model --output_format=tfjs_graph_model --signature_name=serving_default --saved_model_tags=serve /content/export/saved_model /content/export/{CROP}
meta = {"crop": CROP, "labels": CLASSES, "imageSize": IMG, "input": "RGB pixels 0-255, shape [1,224,224,3]",
        "testAccuracy": round(test_acc, 4), "testPhotos": int(len(y_true)), "trainedWith": "MobileNetV3Large transfer learning"}
json.dump(meta, open(OUT / CROP / "metadata.json", "w"), indent=2)
shutil.make_archive(f"/content/grader_{CROP}", "zip", OUT, CROP)
print(json.dumps(meta, indent=2))
from google.colab import files
files.download(f"/content/grader_{CROP}.zip")

In [ ]:
# 10. (Optional) Try your own photo
from google.colab import files
from PIL import Image
for name in files.upload():
    img = Image.open(name).convert("RGB")
    s = min(img.size); l, t = (img.width - s) // 2, (img.height - s) // 2
    x = np.array(img.crop((l, t, l + s, t + s)).resize((IMG, IMG)), dtype="float32")[None]
    p = model.predict(x, verbose=0)[0]
    print(name, "→", CLASSES[p.argmax()], f"{p.max():.0%}", dict(zip(CLASSES, np.round(p, 3))))